# Module 20c: Demand forecasting agent utility development

## Goal

1. Create a stored procedure for forecast generation that can be called by the demand planner agent
2. Create a stored procedure that can update the demand forecast for a specific product


## 1. Set up



In [1]:
PROJECT_ID_LIST=!gcloud config list --format "value(core.project)" 2>/dev/null
PROJECT_ID=PROJECT_ID_LIST[0]
LOCATION="us-central1"
FORECASTING_DS="rscw_fridge_forecasting_ds"

In [2]:
%load_ext google.colab.data_table

### 1.1. Schema for forecast data

In [3]:
%%bigquery --project {PROJECT_ID} --location {LOCATION}

CREATE SCHEMA `rscw_fridge_forecast_ds` OPTIONS (location = 'us-central1');

Query is running:   0%|          |

""


### 1.2. Schema for forecast dataset and table metadata

In [4]:
%%bigquery --project {PROJECT_ID}

CREATE SCHEMA rscw_fridge_forecast_metadata_ds OPTIONS (location = 'us-central1');

Query is running:   0%|          |

""


### 1.3. Create metadata tables

In [5]:
%%bigquery

CREATE OR REPLACE TABLE `rscw_fridge_forecast_metadata_ds.dataset_description`
(
  dataset_description STRING
);

CREATE OR REPLACE TABLE `rscw_fridge_forecast_metadata_ds.dataset_table_relationships`
(
  table_1 STRING,
  table_1_column STRING,
  table_2 STRING,
  table_2_column STRING,
  join_type STRING
);

CREATE OR REPLACE TABLE `rscw_fridge_forecast_metadata_ds.table_column_descriptions`
(
  table_name STRING,
  column_name STRING,
  column_description STRING
);

CREATE OR REPLACE TABLE `rscw_fridge_forecast_metadata_ds.table_descriptions`
(
  name STRING,
  description STRING
);

Query is running:   0%|          |

""


### 1.4. Create demand forecast tables

In [6]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE `rscw_fridge_forecast_ds.demand_forecast_history`
(
  forecast_id STRING,
  forecast_run_time DATETIME,
  generated_by STRING,
  item_number STRING,
  item_name STRING,
  location_id STRING,
  omni_item_id STRING,
  forecast_timestamp TIMESTAMP,
  forecast_value FLOAT64,
  confidence_level FLOAT64,
  prediction_interval_lower_bound FLOAT64,
  prediction_interval_upper_bound FLOAT64,
  ai_forecast_status STRING

);

CREATE TABLE IF NOT EXISTS `rscw_fridge_forecast_ds.demand_forecast`
(
  forecast_id STRING,
  forecast_run_time DATETIME,
  generated_by STRING,
  item_number STRING,
  item_name STRING,
  location_id STRING,
  omni_item_id STRING,
  forecast_timestamp TIMESTAMP,
  forecast_value FLOAT64,
  confidence_level FLOAT64,
  prediction_interval_lower_bound FLOAT64,
  prediction_interval_upper_bound FLOAT64,
  ai_forecast_status STRING
);

CREATE OR REPLACE TABLE `rscw_fridge_forecast_ds.procedure_error_log` (
  error_time DATETIME,
  procedure_name STRING,
  error_message STRING,
  statement_text STRING
);

Query is running:   0%|          |

""


### 1.5. Create demand forecast override configs & demand signal tables

In [7]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}


CREATE OR REPLACE TABLE `rscw_fridge_forecast_ds.forecast_override_configs`
(
  item_number STRING,
  omni_item_id STRING,
  demand_surge_multiplier FLOAT64,
  demand_slump_multiplier FLOAT64,
  last_update_date DATE,
  last_updated_by STRING

);

INSERT INTO rscw_fridge_forecast_ds.forecast_override_configs(item_number,
  omni_item_id,
  demand_surge_multiplier,
  demand_slump_multiplier,
  last_update_date,
  last_updated_by)
SELECT distinct item_number, omni_item_id,1.5,0.75,current_date,'Demand Planner'
from rscw_fridge_ds.product_master where is_active='Y';

select * from rscw_fridge_forecast_ds.forecast_override_configs limit 2;

Query is running:   0%|          |

Downloading:   0%|          |

,item_number,omni_item_id,demand_surge_multiplier,demand_slump_multiplier,last_update_date,last_updated_by
0,RS65DG54R3S9,RS65DG54R3S9-EF,1.5,0.75,2026-01-28,Demand Planner
1,FRWW4543AS,FRWW4543AS,1.5,0.75,2026-01-28,Demand Planner


### 1.6. Create log table for activity logging

In [8]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}


CREATE OR REPLACE TABLE `rscw_fridge_forecast_ds.demand_signal_log`
(
  item_number STRING,
  omni_item_id STRING,
  signal_indicator STRING, --SURGE / SLUMP
  comment STRING,
  signal_datetime DATETIME,
  notified_by STRING
);

CREATE OR REPLACE TABLE `rscw_fridge_forecast_ds.forecast_activity_log`
(
  item_number STRING,
  omni_item_id STRING,
  forecast_id STRING,
  activity_type STRING,
  execution_date DATETIME,
  executed_by STRING
);



Query is running:   0%|          |

""


## 2. Curate data for forecasting

In [9]:
%%bigquery --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE VIEW
    `rscw_fridge_forecast_ds.sales_history_vw` AS
SELECT
    t.transaction_id,
    t.location_id,
    t.customer_id,
    t.transaction_status,
    t.transaction_date,
    t.payment_type,
    t.payment_total_dollar,
    ti.item_number,
    ti.omni_item_id,
    ti.quantity,
    pm.description,
    pm.appliance_sub_type,
    pm.brand,
    ti.price,
    ti.line_item_total,
    cm.city
  FROM
    `rscw_fridge_ds.pos_transactions` AS t
    INNER JOIN `rscw_fridge_ds.pos_transaction_items` AS ti ON t.transaction_id = ti.transaction_id
    INNER JOIN `rscw_fridge_ds.product_master` AS pm ON ti.omni_item_id = pm.omni_item_id
    INNER JOIN `rscw_fridge_ds.customer_master` AS cm ON t.customer_id = cm.customer_id

Query is running:   0%|          |

""


In [10]:
%%bigquery --project {PROJECT_ID} --location {LOCATION}

Select * from
    `rscw_fridge_forecast_ds.sales_history_vw` LIMIT 2

Query is running:   0%|          |

Downloading:   0%|          |

,transaction_id,location_id,customer_id,transaction_status,transaction_date,payment_type,payment_total_dollar,item_number,omni_item_id,quantity,description,appliance_sub_type,brand,price,line_item_total,city
0,8087508e-306a-41a7-8c1d-4432c5fd8caa,NAP-IL-ST,47166f88862447ea5f8e078b2aaefb46,SALE,2025-09-08,CASH,1699.000000000,LRDCS2603S,LRDCS2603S,1,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,bottom freezer,LG,1699.000000000,1699.000000000,Naperville
1,c84fa9aa-b43b-4ab1-be77-f808b4a4c1c3,NAP-IL-ST,46490458d353247266448e9bdc7bd115,SALE,2025-09-08,CASH,1699.000000000,LRDCS2603S,LRDCS2603S,1,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,bottom freezer,LG,1699.000000000,1699.000000000,Schaumburg


In [11]:
%%bigquery --project {PROJECT_ID} --location {LOCATION}


-- SALES AGGREGATED BY DATE, STORE, ITEM_NM
CREATE OR REPLACE VIEW
  `rscw_fridge_forecast_ds.aggr_sales_by_item_vw` AS (
WITH top_sellers AS(
      SELECT
        item_number,
        omni_item_id,
        description as item_nm,
        SUM(quantity) AS total_quantity
    FROM
        `rscw_fridge_forecast_ds.sales_history_vw`
    GROUP BY
        item_number,
        omni_item_id,
        description
    ORDER BY total_quantity DESC
    LIMIT 25
)
SELECT
    transaction_date,
    location_id,
    item_number,
    omni_item_id,
    description as item_nm,
     SUM(quantity) AS total_quantity
FROM
    `rscw_fridge_forecast_ds.sales_history_vw`
GROUP BY transaction_date, location_id,
    item_number,
    omni_item_id,
    description
HAVING
    transaction_date BETWEEN '2025-01-01' AND '2026-01-31'
    AND omni_item_id IN (SELECT omni_item_id FROM top_sellers)
);

Query is running:   0%|          |

""


In [12]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

select * from rscw_fridge_forecast_ds.aggr_sales_by_item_vw where omni_item_id='LRDCS2603S' LIMIT 2

Query is running:   0%|          |

Downloading:   0%|          |

,transaction_date,location_id,item_number,omni_item_id,item_nm,total_quantity
0,2025-09-03,SCH-IL-ST,LRDCS2603S,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,9
1,2025-08-19,CHI-IL-ST,LRDCS2603S,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,9


## 3. Create a stored procedure for forecasting as part of scheduled process or on-demand

In [13]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE PROCEDURE `rscw_fridge_forecast_ds.run_demand_forecast`(p_invoker STRING)
BEGIN
  -- 1. Declare and initiative variables
  DECLARE v_forecast_id STRING DEFAULT GENERATE_UUID();
  DECLARE v_forecast_datetime DATETIME DEFAULT CURRENT_DATETIME;

  -- 2. Save existing forecast to history table before overwriting
  -- Ensure demand_forecast_history exists and has a compatible schema
  INSERT INTO `rscw_fridge_forecast_ds.demand_forecast_history`
  SELECT * FROM `rscw_fridge_forecast_ds.demand_forecast`;

  TRUNCATE TABLE rscw_fridge_forecast_ds.demand_forecast;

  -- 3. Run the latest forecast
  -- Overwrites the current forecast table with fresh 30-day predictions
  BEGIN
    INSERT INTO rscw_fridge_forecast_ds.demand_forecast
    SELECT DISTINCT v_forecast_id,v_forecast_datetime as forecast_run_time,p_invoker as generated_by,'' as item_number, '' as item_name, location_id ,omni_item_id,
    forecast_timestamp,forecast_value,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
    FROM
      AI.FORECAST(
        (
          SELECT *
          FROM rscw_fridge_forecast_ds.aggr_sales_by_item_vw
        ),
        horizon => 30,
        confidence_level => 0.95,
        timestamp_col => 'transaction_date',
        data_col => 'total_quantity',
        id_cols => ['location_id', 'omni_item_id']
        )
      ;

    -- 4. Enrich the forecast with Product Master details
    -- Joins on item IDs to fill in readable names and numbers
    MERGE `rscw_fridge_forecast_ds.demand_forecast` AS DF
    USING `rscw_fridge_ds.product_master` AS PM
    ON DF.omni_item_id = PM.omni_item_id
    WHEN MATCHED THEN
      UPDATE SET
        DF.item_number = PM.item_number,
        DF.item_name = PM.description;

    --5. Log activity to table
    INSERT INTO `rscw_fridge_forecast_ds.forecast_activity_log`(item_number,
    omni_item_id,forecast_id,activity_type,execution_date,executed_by)
    select null,null,v_forecast_id,'FORECAST_RUN',v_forecast_datetime,p_invoker;


  EXCEPTION WHEN ERROR THEN
    INSERT INTO `rscw_fridge_forecast_ds.procedure_error_log` (
      error_time,
      procedure_name,
      error_message,
      statement_text
    )
    VALUES (
      v_forecast_datetime,
      'run_demand_forecast',
      @@error.message,          -- The actual error message
      @@error.statement_text    -- The specific SQL that failed
    );

    RAISE USING MESSAGE = FORMAT("Procedure failed: %s", @@error.message);
  END;

END;

Query is running:   0%|          |

""


#### Lets test the forecast

In [14]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CALL rscw_fridge_forecast_ds.run_demand_forecast('TESTING');

Query is running:   0%|          |

""


In [15]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

select * from rscw_fridge_forecast_ds.demand_forecast where omni_item_id='LRDCS2603S' and DATE(forecast_timestamp)='2026-02-22'

Query is running:   0%|          |

Downloading:   0%|          |

,forecast_id,forecast_run_time,generated_by,item_number,item_name,location_id,omni_item_id,forecast_timestamp,forecast_value,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
0,1e3d60f7-a8b3-44b4-af3e-10c526b9d078,2026-01-28 15:40:33.776369,TESTING,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,CHI-IL-ST,LRDCS2603S,2026-02-22 00:00:00+00:00,10.645513,0.95,6.625882,14.665143,
1,1e3d60f7-a8b3-44b4-af3e-10c526b9d078,2026-01-28 15:40:33.776369,TESTING,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,NAP-IL-ST,LRDCS2603S,2026-02-22 00:00:00+00:00,10.047140,0.95,6.739970,13.354311,
2,1e3d60f7-a8b3-44b4-af3e-10c526b9d078,2026-01-28 15:40:33.776369,TESTING,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,SCH-IL-ST,LRDCS2603S,2026-02-22 00:00:00+00:00,10.680650,0.95,7.748474,13.612826,


## 4. Create stored procedure for on-demand forecast update for a specific product

In [16]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}


CREATE OR REPLACE PROCEDURE `rscw_fridge_forecast_ds.update_item_demand_forecast`(p_omni_item_id STRING, p_adjustment_type STRING, p_invoker STRING)
BEGIN
  -- 1. Declare and initialize variables
  DECLARE v_forecast_batch_id STRING DEFAULT GENERATE_UUID();
  DECLARE v_forecast_datetime DATETIME DEFAULT CURRENT_DATETIME;

  --2. Get the multipler for the demand adjustment
  DECLARE v_demand_multiplier FLOAT64;

  SET v_demand_multiplier = (
    SELECT (CASE WHEN p_adjustment_type='SURGE' THEN demand_surge_multiplier ELSE demand_slump_multiplier END)
    FROM `rscw_fridge_forecast_ds.forecast_override_configs`
    WHERE omni_item_id = p_omni_item_id
  );

  BEGIN
    -- 3. Archive only the product being updated
    -- This keeps your history clean and specific
    INSERT INTO `rscw_fridge_forecast_ds.demand_forecast_history`
    SELECT * FROM `rscw_fridge_forecast_ds.demand_forecast`
    WHERE (omni_item_id = p_omni_item_id);

    -- 4. Update the existing forecast to accomodate the demand surge
    UPDATE `rscw_fridge_forecast_ds.demand_forecast`
    SET generated_by = 'AGENT_OVERRIDE',
        forecast_value = forecast_value * v_demand_multiplier,
        forecast_id = v_forecast_batch_id,
        forecast_run_time = v_forecast_datetime
    WHERE omni_item_id = p_omni_item_id;

    --5. Log the activity into the activty log table
    INSERT INTO `rscw_fridge_forecast_ds.forecast_activity_log`(item_number,
    omni_item_id,forecast_id,activity_type,execution_date,executed_by)
    select null,p_omni_item_id,v_forecast_batch_id,'FORECAST_UPDATE',v_forecast_datetime,p_invoker;

  EXCEPTION WHEN ERROR THEN
    INSERT INTO `rscw_fridge_forecast_ds.procedure_error_log` (error_time, procedure_name, error_message)
    VALUES (v_forecast_datetime, 'update_item_demand_forecast', @@error.message);
    RAISE USING MESSAGE = @@error.message;
  END;
END;

Query is running:   0%|          |

""


In [17]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

--The second parameter is a multiplier to increase or decrease existing demand
CALL rscw_fridge_forecast_ds.update_item_demand_forecast('LRDCS2603S','SURGE','DEMAND PLANNER AGENT');
--Previous value was 10.77

Query is running:   0%|          |

""


In [18]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

SELECT * FROM rscw_fridge_forecast_ds.demand_forecast where omni_item_id='LRDCS2603S' and DATE(forecast_timestamp)='2026-02-01'


Query is running:   0%|          |

Downloading:   0%|          |

,forecast_id,forecast_run_time,generated_by,item_number,item_name,location_id,omni_item_id,forecast_timestamp,forecast_value,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
0,c8605f78-0921-4082-9d1d-78ac91de4675,2026-01-28 15:41:03.428873,AGENT_OVERRIDE,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,CHI-IL-ST,LRDCS2603S,2026-02-01 00:00:00+00:00,15.700257,0.95,6.983012,13.950664,
1,c8605f78-0921-4082-9d1d-78ac91de4675,2026-01-28 15:41:03.428873,AGENT_OVERRIDE,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,NAP-IL-ST,LRDCS2603S,2026-02-01 00:00:00+00:00,15.368890,0.95,6.974386,13.517468,
2,c8605f78-0921-4082-9d1d-78ac91de4675,2026-01-28 15:41:03.428873,AGENT_OVERRIDE,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,SCH-IL-ST,LRDCS2603S,2026-02-01 00:00:00+00:00,15.614917,0.95,7.422993,13.396897,


In [19]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

SELECT * FROM rscw_fridge_forecast_ds.demand_forecast_history where omni_item_id='LRDCS2603S' and DATE(forecast_timestamp)='2026-02-22'
UNION ALL
SELECT * FROM rscw_fridge_forecast_ds.demand_forecast where omni_item_id='LRDCS2603S' and DATE(forecast_timestamp)='2026-02-22'
ORDER BY location_id,forecast_run_time desc

Query is running:   0%|          |

Downloading:   0%|          |

,forecast_id,forecast_run_time,generated_by,item_number,item_name,location_id,omni_item_id,forecast_timestamp,forecast_value,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
0,c8605f78-0921-4082-9d1d-78ac91de4675,2026-01-28 15:41:03.428873,AGENT_OVERRIDE,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,CHI-IL-ST,LRDCS2603S,2026-02-22 00:00:00+00:00,15.968269,0.95,6.625882,14.665143,
1,1e3d60f7-a8b3-44b4-af3e-10c526b9d078,2026-01-28 15:40:33.776369,TESTING,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,CHI-IL-ST,LRDCS2603S,2026-02-22 00:00:00+00:00,10.645513,0.95,6.625882,14.665143,
2,c8605f78-0921-4082-9d1d-78ac91de4675,2026-01-28 15:41:03.428873,AGENT_OVERRIDE,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,NAP-IL-ST,LRDCS2603S,2026-02-22 00:00:00+00:00,15.070710,0.95,6.739970,13.354311,
3,1e3d60f7-a8b3-44b4-af3e-10c526b9d078,2026-01-28 15:40:33.776369,TESTING,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,NAP-IL-ST,LRDCS2603S,2026-02-22 00:00:00+00:00,10.047140,0.95,6.739970,13.354311,
4,c8605f78-0921-4082-9d1d-78ac91de4675,2026-01-28 15:41:03.428873,AGENT_OVERRIDE,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,SCH-IL-ST,LRDCS2603S,2026-02-22 00:00:00+00:00,16.020975,0.95,7.748474,13.612826,
5,1e3d60f7-a8b3-44b4-af3e-10c526b9d078,2026-01-28 15:40:33.776369,TESTING,LRDCS2603S,26 cu. ft. 33 Inch Wide Bottom Freezer Refrige...,SCH-IL-ST,LRDCS2603S,2026-02-22 00:00:00+00:00,10.680650,0.95,7.748474,13.612826,


## 5. Create a stored procedure for logging demand signals received

In [20]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}


CREATE OR REPLACE PROCEDURE `rscw_fridge_forecast_ds.log_demand_signal`(p_item_number STRING, p_omni_item_id  STRING, p_signal_indicator  STRING, p_comment  STRING, p_notified_by  STRING)
BEGIN

  DECLARE v_signal_datetime DATETIME DEFAULT CURRENT_DATETIME;

  BEGIN

    INSERT INTO rscw_fridge_forecast_ds.demand_signal_log(item_number, omni_item_id, signal_indicator, comment, notified_by, signal_datetime)
    select p_item_number, p_omni_item_id, p_signal_indicator, p_comment, p_notified_by,v_signal_datetime;

  EXCEPTION WHEN ERROR THEN
    INSERT INTO rscw_fridge_forecast_ds.procedure_error_log(error_time, procedure_name, error_message)
    VALUES (v_forecast_datetime, 'log_demand_signal', @@error.message);
    RAISE USING MESSAGE = @@error.message;
  END;
END;

Query is running:   0%|          |

""


In [21]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CALL rscw_fridge_forecast_ds.log_demand_signal('LRDCS2603S','LRDCS2603S','SURGE','This product was endorsed by a celebrity on social media and there is a demand surge noticed with our competitors','MARKET_INTELLIGENCE_AGENT');

Query is running:   0%|          |

""


In [22]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

select * from rscw_fridge_forecast_ds.demand_signal_log

Query is running:   0%|          |

Downloading:   0%|          |

,item_number,omni_item_id,signal_indicator,comment,signal_datetime,notified_by
0,LRDCS2603S,LRDCS2603S,SURGE,This product was endorsed by a celebrity on so...,2026-01-28 15:41:14.710826,MARKET_INTELLIGENCE_AGENT


## 6. Run Data Insights scans on the rscw_fridge_forecast_ds dataset

#### **THIS STEP IS VERY IMPORTANT FOR AGENT ACCURACY FOR NL QUERIES**

Switch to the notebook-Module_20b_Generic_Agentic_Grounding_Prework and run the notebook after going to the last cell of the notebook and updating the dataset id to `rscw_fridge_forecast_ds`

## This concludes the agent utility development, proceed back to the user manual.